# PBT #1 checkpoint: evaluation and inference video

Loads **only the rank-01** checkpoint under `best_pbt_checkpoints_top5/` (best eval_score in the manifest), runs the same **deterministic** evaluation as training (`evaluate_current_model` in `ray_pbt_train.py`), then shows one **inference rollout** as an inline animation (same rendering pattern as `VideoProgressCallback._record_and_show` in `unit1 - Lunar Lander agent_v3.ipynb`).

**Eval (same as Unit 1):** `evaluate_policy` for **`periodic_eval_episodes`** (usually 10), **`deterministic=True`**. Prints **`mean_reward +/- std_reward`** across those episodes (std = spread across episodes). **Leaderboard-style score** = **mean_reward − std_reward**. Env: one `VecNormalize` eval env via `make_eval_vec_env_synced` (`training=False`, `norm_reward=False`). `best_eval_score_so_far` in `trainer_state.json` may differ from a fresh eval (RNG / best-so-far during training).

**Video:** one **stochastic** episode for display only — **not** the 10-episode eval score.

**Setup:** open this notebook from the repo root (or set `REPO_ROOT` in the next cell). Use the project `.venv` like the Unit 1 notebook. For headless servers, install `xvfb` and use `pyvirtualdisplay` as in Unit 1 so `rgb_array` rendering works.

In [5]:
import os
import warnings

# Silence pygame → setuptools pkg_resources deprecation (main process + SubprocVecEnv workers).
os.environ.setdefault("PYTHONWARNINGS", "ignore::UserWarning:pygame.pkgdata")
warnings.filterwarnings(
    "ignore",
    message=".*pkg_resources is deprecated.*",
    category=UserWarning,
    module="pygame.pkgdata",
)

%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.animation as animation
import matplotlib.pyplot as plt
from IPython.display import HTML, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "ray_pbt_train.py").is_file():
    raise RuntimeError(
        "Set the notebook working directory to the RL-LunarLander repo root "
        "(File → Open Folder), or set REPO_ROOT manually."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from lunar_rl_common import DEFAULT_ENV_ID, make_eval_vec_env_synced
from ray_pbt_train import evaluate_current_model, load_trial_checkpoint

CHECKPOINT_ROOT = REPO_ROOT / "best_pbt_checkpoints_top5"

In [6]:
def sorted_checkpoint_dirs(root: Path) -> list[Path]:
    dirs = [p for p in root.iterdir() if p.is_dir() and (p / "model.zip").is_file()]
    return sorted(dirs, key=lambda p: p.name)


def collect_rollout_frames(model, train_venv, seed: int, env_id: str | None):
    """One deterministic episode; RGB frames via env.render (matches VideoProgressCallback)."""
    eval_venv = make_eval_vec_env_synced(train_venv, seed, env_id)
    try:
        eval_venv.seed(seed)
        obs = eval_venv.reset()
        frames = []
        fr = eval_venv.env_method("render")[0]
        if fr is not None:
            frames.append(fr)
        total_reward = 0.0
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = eval_venv.step(action)
            total_reward += float(reward[0])
            fr = eval_venv.env_method("render")[0]
            if fr is not None:
                frames.append(fr)
            if done[0]:
                break
        return frames, total_reward
    finally:
        eval_venv.close()


def display_rollout_video(frames, title: str, interval_ms: int = 33) -> None:
    if not frames:
        print(f"{title}: no frames (check display / rgb_array rendering).")
        return
    fig, ax = plt.subplots(figsize=(6, 4), dpi=72)
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=0.9, bottom=0.05)
    fig.suptitle(title, fontsize=11)
    im = ax.imshow(frames[0])

    def update(i):
        im.set_array(frames[i])
        return (im,)

    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=interval_ms, blit=False
    )
    html = anim.to_jshtml()
    plt.close(fig)
    display(HTML(html))


_all = sorted_checkpoint_dirs(CHECKPOINT_ROOT)
checkpoint_dirs = _all[:1]
if not checkpoint_dirs:
    raise RuntimeError(f"No checkpoint with model.zip under {CHECKPOINT_ROOT}")
print(
    f"Using rank #1 only ({checkpoint_dirs[0].name}); skipped {len(_all) - 1} other(s)."
)

for ckpt_dir in checkpoint_dirs:
    print("\n" + "=" * 72)
    print(ckpt_dir.name)
    state_path = ckpt_dir / "trainer_state.json"
    with open(state_path, encoding="utf-8") as f:
        tr_state = json.load(f)
    merged = tr_state["current_config"]
    seed = int(tr_state["seed"])
    n_eval = int(merged["base"]["periodic_eval_episodes"])
    stored_best = tr_state.get("best_eval_score_so_far")

    model, train_env, _loaded = load_trial_checkpoint(
        str(ckpt_dir), merged, env_id=DEFAULT_ENV_ID
    )
    try:
        mean_r, std_r, score = evaluate_current_model(
            model, train_env, seed, n_eval, env_id=DEFAULT_ENV_ID
        )
        print(
            f"evaluate_policy (n_eval_episodes={n_eval}, deterministic=True) — Unit 1 style:"
        )
        print(f"  mean_reward={mean_r:.2f} +/- {std_r:.2f}")
        print(
            f"  eval_score (mean_reward - std_reward, leaderboard)={score:.2f}"
        )
        if stored_best is not None:
            print(
                f"  trainer_state.json best_eval_score_so_far (during PBT): "
                f"{float(stored_best):.2f}"
            )
        print(
            "  (VecNormalize synced obs; norm_reward=False; training=False.)"
        )
        frames, ep_ret = collect_rollout_frames(
            model, train_env, seed, DEFAULT_ENV_ID
        )
        print(
            f"video replay (1 episode only — not the eval score above): return={ep_ret:.2f}"
        )
        display_rollout_video(frames, title=ckpt_dir.name)
    finally:
        train_env.close()

Using rank #1 only (rank01_ts5963776_mean301.37_eval293.77_trial1e06f_00012_srccheckpoint_000009); skipped 4 other(s).

rank01_ts5963776_mean301.37_eval293.77_trial1e06f_00012_srccheckpoint_000009
evaluate_policy (n_eval_episodes=10, deterministic=True) — Unit 1 style:
  mean_reward=301.37 +/- 7.60
  eval_score (mean_reward - std_reward, leaderboard)=293.77
  trainer_state.json best_eval_score_so_far (during PBT): 293.77
  (VecNormalize synced obs; norm_reward=False; training=False.)
video replay (1 episode only — not the eval score above): return=262.80


In [ ]:
import subprocess

from huggingface_hub import notebook_login

notebook_login()
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

In [ ]:

from huggingface_sb3 import package_to_hub

## repo_id is the id of the model repository from the Hugging Face Hub (repo_id = {organization}/{repo_name} for instance username/ppo-LunarLander-v3
repo_id = "ntitz19/ppo-LunarLander-v3"

# TODO: Define the name of the environment

eval_env = make_eval_vec_env_with_stats(vecnormalize_path, SEED, env_id)

# TODO: Define the model architecture we used
model_architecture = "PPO"

## TODO: Define the commit message
commit_message = "Upload PPO LunarLander-v3 MlpPolicy vector-obs agent"

_sig = inspect.signature(package_to_hub)
_kwargs = dict(
    model=model,
    model_name=model_name,
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message,
)
if "n_eval_episodes" in _sig.parameters:
    _kwargs["n_eval_episodes"] = 10
# If your huggingface_sb3 is older and lacks n_eval_episodes, the default episode count applies.

package_to_hub(**_kwargs)
eval_env.close()

# The course progress checker looks for "LunarLander-v2" tag, but the env is v3.
# Patch the model card to include the v2 tag so the checker recognizes it.
from huggingface_hub import ModelCard

card = ModelCard.load(repo_id)
if "LunarLander-v2" not in card.data.tags:
    card.data.tags.append("LunarLander-v2")
    card.push_to_hub(
        repo_id, commit_message="Add LunarLander-v2 tag for course certification compatibility"
    )
